# Stage 17 · Finish and score the three personal-scene runs

Attach the private **personal-input.zip**, **personal-s42-stage16-review.zip** and the **Stage 4 offline LPIPS kit**. Select **GPU T4 ×2** and **Run All**. Expect all three models to restore at 10,000. Each uses its original seed, optimiser and random states. The first 50,000 updates use the original decay and the final 25,000 hold the final learning rate.

With 5.03–5.14 updates/s measured on this capture, the staggered two-GPU waves project to about 5½ hours of training. Allow **5¾–6½ hours wall time** including scoring and exports. The notebook saves a portable snapshot after each completed wave. Download the **step30000**, **step50000** and **step75000** backups when offered. The scoring cell renders the same 12 held-out images for each model and the eight training views for each sparse arm.

Send **personal-s42-stage17-review.zip**; keep the outputs private. If interrupted, attach the latest successful Stage 17 checkpoint or evaluation snapshot alongside the original two kits in a fresh session. Send the traceback and latest backup if a check fails. Videos come after score review.

In [ ]:
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
import time
import zipfile

import torch
from IPython.display import HTML, display
from PIL import Image

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
root = outputs / "recovery-stage17"
root.mkdir(parents=True, exist_ok=True)
if "stage17_temp" not in globals():
    stage17_temp = TemporaryDirectory(prefix="sparse-stage17-")
cache = Path(stage17_temp.name)
def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def unpack(path):
    folder = cache / hashlib.sha256(str(path).encode()).hexdigest()
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not (folder / name).resolve().is_relative_to(folder.resolve()):
                raise ValueError(f"Unexpected member {name}")
            archive.extract(name, folder)
    return folder

def kit(marker, filename):
    roots = sorted(inputs.rglob(marker))
    if roots:
        return roots[0].parent
    archives = sorted(inputs.rglob(filename))
    if not archives:
        raise FileNotFoundError(f"Attach {filename} or its expanded dataset.")
    return unpack(archives[0])

repo = kit("RECOVERY_PERSONAL.txt", "personal-input.zip")
metric_kit = kit("RECOVERY_LPIPS.txt", "stage4-lpips.zip")

camera_ids = [1, 4, 6, 13, 25, 41, 69, 74]
data_hash = 'd39c6ff67262904660cf715ea3ba6cdbd17ae3a0cf07a1b83a917cbb50dbe8ae'
code_hash = '4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5'
work = root
arms = ('dense-vanilla', 'sparse-vanilla', 'sparse-regularized')
view_counts = {'dense-vanilla': 64, 'sparse-vanilla': 8, 'sparse-regularized': 8}
start_shas = {'dense-vanilla': '98d992948ffb77b929e2e1cdfc03b9c33111a041e5a00886ebbd225e2689a4a8', 'sparse-vanilla': '9ad88bd116bf624628fa386bd4a09e3643216b493d95dbe51e138f0260bdbda0', 'sparse-regularized': '00c6cefbfa7f13facb0f28ab9a7a2a00d6b874da5935b237cff5c17e0d8e1947'}
start_ids = {'dense-vanilla': 'c23621d6ebb84aad9b02399c5f68d962', 'sparse-vanilla': '50aaf67f3ca74053a65beedfa5eb2fda', 'sparse-regularized': 'dd192d6510d84ec59ddec68f1ddc8c50'}
stage16_receipt = '44113c4ffc5cce0b87ed87de291c2052455edb5a7048781d0ecc9f8d3e807919'
controls = {'dense-vanilla': {'black': 5.432834960955936, 'white': 5.386597168667672, 'constant': 16.043457667854355}, 'sparse-vanilla': {'black': 5.7147920218809745, 'white': 5.300954393201087, 'constant': 16.41939352983131}, 'sparse-regularized': {'black': 5.7147920218809745, 'white': 5.300954393201087, 'constant': 16.41939352983131}}
boundary, endpoint = 50_000, 75_000
device_count = torch.cuda.device_count()
if device_count < 2:
    raise RuntimeError("Select GPU T4 ×2 and restart the notebook to run two workers concurrently.")
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = [v.strip() for v in visible.split(",")][:2] if visible else ["0", "1"]
assert len(set(devices)) == 2
print("Workers:", [torch.cuda.get_device_name(i) for i in range(2)])
sys.path.insert(0, str(repo))
from recovery.data import freeze, verify_manifest
from recovery.health import dead_field
from recovery.media import digest, model_fingerprint
from recovery.training import fingerprint, implementation_hash, load_state, make_model, settings, write_json

assert implementation_hash() == code_hash
manifest = work / "manifests/personal.json"
manifest.parent.mkdir(exist_ok=True)
frozen = verify_manifest(repo/"manifests/personal.json",repo/"data/personal")
assert frozen["data_hash"]==data_hash and frozen["splits"]["sparse"]==camera_ids
write_json(manifest,frozen)
cfg = settings("main", frozen["camera"])
plan = dict(cfg, steps=75_000, lr=cfg["lr_final"], prefix_config=cfg, prefix_updates=50_000,
            schedule="50k-exponential-then-25k-hold-v1")
assert digest(cfg) == "1ccbfe38d7b2c561c69ecbc7c2b609f0aa77818523cd7d39f7d717a524828b9e"
assert digest(plan) == "280e2b9ada481e2844b471404f1d99eac4fff5a35bc069cc4385327b3a1cc3a9"
print("Verified 64 dense cameras, the frozen eight sparse cameras and the matched two-phase schedule.")
score_sources = {
    "recovery/media.py": "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af",
    "sparse_nerf/evaluate.py": "eb2a0c14ab7aac4de41fc5de9e2e2316ba902e08a14cc2b1a06d7616171e58ca",
}
for name, expected in score_sources.items():
    assert hashlib.sha256((repo / name).read_text(encoding="utf-8").encode()).hexdigest() == expected
from sparse_nerf.data import SceneDataset
from recovery.media import comparison_settings, psnr, rgb_uint8
test = SceneDataset(manifest, "test", cfg["downscale"])
assert len(frozen["splits"]["dense"]) == 64 and len(test) == 12
expected_comparison = comparison_settings(test, dict(training_config=plan, data_hash=data_hash))
expected_comparison["lpips_weights_sha256"] = "913e6288d265ca661f26fe0426b840bc69bafcf9fcda0fb2c99586891562ea9e"
targets = hashlib.sha256()
white_views = []
for i in range(len(test)):
    target = test.image(i)
    targets.update(rgb_uint8(target).tobytes())
    white_views.append(psnr(torch.ones_like(target), target))
expected_comparison["target_hash"] = targets.hexdigest()
print("Personal test targets verified. White PSNR", statistics.mean(white_views), flush=True)

In [ ]:
def relative(arm):
    return Path("runs/personal") / arm / "seed-42"

def checked(run, arm):
    meta = read_json(run / "run.json")
    assert (meta["scene"], meta["protocol"], meta["seed"], meta["train_views"], meta["data_hash"],
            meta["downscale"], meta["device_type"]) == ("personal", arm, 42, view_counts[arm], data_hash, 4, "cuda")
    hold = meta["training_config"] == plan
    if hold:
        lineage = meta["lineage"]
        assert (lineage["parent_run_id"], lineage["parent_step"]) == (start_ids[arm], boundary)
        parent_sha = lineage["parent_checkpoint_sha256"]
        assert len(bytes.fromhex(parent_sha)) == 32
        assert meta["run_id"] == hashlib.sha256(f"{parent_sha}/{plan['schedule']}".encode()).hexdigest()[:32]
    else:
        assert meta["training_config"] == cfg and meta["run_id"] == start_ids[arm]
    state = load_state(run, meta)
    step = state["step"]
    low, high = (boundary, endpoint) if hold else (10000, boundary)
    assert low <= step <= high
    slots = state["optimizer"]["state"]
    assert len(slots) == 48 and all(entry["step"].item() == step for entry in slots.values())
    tensors = [*state["model"].values(), *(e[k] for e in slots.values() for k in ("exp_avg", "exp_avg_sq"))]
    assert all(t.isfinite().all().item() for t in tensors)
    assert all(isinstance(state[k], torch.Tensor) and state[k].numel() for k in ("cpu_rng", "cuda_rng"))
    lr = cfg["lr_final"] if hold else cfg["lr"] * (cfg["lr_final"]/cfg["lr"])**((step-1)/(boundary-1))
    assert all(math.isclose(g["lr"], lr, rel_tol=1e-10) for g in state["optimizer"]["param_groups"])
    return step, hold

def healthy(run, step):
    health = read_json(run / "health.json")
    fine = health["fine"]
    initial = read_json(run / "initial-health.json")
    arm=read_json(run/"run.json")["protocol"]
    ref=controls[arm]
    return (health["region"]=="central rectangle" and fine["region_psnr"]>max(ref["constant"],ref["black"]+3) and health["step"] == step and not dead_field(health)
            and fine["region_psnr"] > health["white_region_psnr"] + 3
            and fine["region_mse"] < initial["fine"]["region_mse"] and fine["region_opacity"] > .02)

sources = {arm: [] for arm in arms}
def candidate(run, arm, local, files=None):
    step, hold = checked(run, arm)
    ck = sha(run / "checkpoint.pt")
    assert step != 10000 or ck == start_shas[arm]
    scored = (run / "eval/metrics.json").is_file() and (local or "eval/metrics.json" in files)
    sources[arm].append((step, hold, scored, local, ck, run, files))
    return step, ck

for stage, pattern in [(16, "personal-s42-stage16-review.zip"), (17, "personal-s42-stage17-*.zip")]:
    marker = f"stage{stage}-review.json"
    folders = [p.parent for p in inputs.rglob(marker)]
    folders += [unpack(p) for p in sorted(inputs.rglob(pattern)) if not p.name.endswith("-diagnostics.zip")]
    for folder in folders:
        receipt = read_json(folder / marker)
        if receipt.get("tag") == "diagnostics":
            continue
        if stage == 16:
            assert sha(folder/marker) == stage16_receipt and receipt["review_step"]==10000 and not receipt["process_errors"]
        else:
            assert receipt["stage16_receipt"] == stage16_receipt and receipt["start_checkpoints"] == start_shas
        assert receipt["schedule_hash"] == digest(plan) and receipt["data_hash"] == data_hash
        for name, expected in receipt["sha256"].items():
            assert not Path(name).is_absolute() and ".." not in Path(name).parts
            p = (folder / name).resolve()
            assert p.is_relative_to(folder.resolve()) and sha(p) == expected
        for arm in arms:
            rec = receipt["arms"][arm]
            rel = relative(arm)
            members = {Path(n).relative_to(rel).as_posix() for n in receipt["sha256"] if Path(n).is_relative_to(rel)}
            assert members >= {"run.json", "checkpoint.pt", "initial-health.json", "health.json", "training.jsonl"}
            src = folder / rel
            step, ck = candidate(src, arm, False, members)
            assert (step, ck, read_json(src / "run.json")["run_id"]) == (rec["step"], rec["checkpoint_sha256"], start_ids[arm] if stage==16 else rec["run_id"])
            assert stage != 16 or (step == 10000 and rec["passed"] and healthy(src, step))

paths = {}
for arm in arms:
    parent = work / "instances" / arm
    parent.mkdir(exist_ok=True, parents=True)
    for checkpoint in parent.glob("*/checkpoint.pt"):
        candidate(checkpoint.parent, arm, True)
    if not sources[arm]:
        raise FileNotFoundError("Attach the reviewed Stage 16 review or a Stage 17 checkpoint snapshot.")
    best = max(sources[arm], key=lambda row: row[:4])
    step, hold, scored, local, ck, run, members = best
    if len({r[4] for r in sources[arm] if r[:2] == (step, hold)}) != 1:
        raise ValueError(f"Conflicting {arm} checkpoints at {step}. Use the intended snapshot in a fresh session.")
    if not local:
        with TemporaryDirectory(dir=parent) as temp:
            staging = Path(temp) / "run"
            for name in members:
                target = staging / name
                target.parent.mkdir(exist_ok=True, parents=True)
                shutil.copyfile(run / name, target)
            run = parent / (ck + ("-scored" if scored else ""))
            staging.rename(run)
    paths[arm] = run
    print(f"Restored {arm} at {step:,} updates. Held-out score present: {scored}")

In [ ]:
# Install the pinned LPIPS wheel in a stage-local directory. Kaggle supplies torch and torchvision.
wheel = metric_kit / "wheels/lpips-0.1.4-py3-none-any.whl"
torch_cache = metric_kit / "torch-cache"
weights = torch_cache / "hub/checkpoints/alexnet-owt-7be5be79.pth"
wheel_hash = sha(wheel)
weight_hash = sha(weights)
assert wheel_hash == "fd537af5828b69d2e6ffc0a397bd506dbc28ca183543617690844c08e102ec5e"
assert weight_hash == "7be5be791159472b1fbf3c69796f7cb30dca7ad8466c2df70058c37116cdee02"
site = work / "metric-packages"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--upgrade",
                "--target", str(site), str(wheel)], check=True)
env = os.environ.copy()
env.update(TORCH_HOME=str(torch_cache), PYTHONPATH=os.pathsep.join([str(site), str(repo), env.get("PYTHONPATH", "")]))
probe = """
import importlib.metadata, json, torch, torchvision
from sparse_nerf.evaluate import lpips_model
from recovery.media import model_fingerprint
net = lpips_model().eval()
with torch.inference_mode():
    sample = torch.zeros(1, 3, 64, 64)
    identical = net(sample, sample).item()
    different = net(sample, sample + 1).item()
assert identical == 0.0 and different > 0 and torch.isfinite(torch.tensor(different))
print(json.dumps(dict(lpips=importlib.metadata.version('lpips'), torchvision=torchvision.__version__,
                      lpips_model_sha256=model_fingerprint(net), identical=identical, different=different)))
"""
probe_output = subprocess.check_output([sys.executable, "-c", probe], cwd=repo, env=env, text=True)
lpips_info = json.loads(probe_output.strip().splitlines()[-1])
assert lpips_info["lpips"] == "0.1.4"
assert lpips_info["lpips_model_sha256"] == "913e6288d265ca661f26fe0426b840bc69bafcf9fcda0fb2c99586891562ea9e"
print("Offline LPIPS-Alex v0.1 ready.", lpips_info)

In [ ]:
def fork_hold(arm):
    parent = paths[arm]
    step, hold = checked(parent, arm)
    if hold:
        return
    assert step == boundary and healthy(parent, step)
    meta = read_json(parent / "run.json")
    state = load_state(parent, meta)
    parent_sha = sha(parent / "checkpoint.pt")
    child_id = hashlib.sha256(f"{parent_sha}/{plan['schedule']}".encode()).hexdigest()[:32]
    lineage = dict(parent_run_id=meta["run_id"], parent_checkpoint_sha256=parent_sha, parent_step=boundary)
    child = dict(meta, run_id=child_id, training_config=plan, lineage=lineage)
    dest = parent.parent / ("hold-" + child_id)
    if not dest.exists():
        with TemporaryDirectory(dir=parent.parent) as temp:
            staging = Path(temp) / "run"
            shutil.copytree(parent, staging, ignore=shutil.ignore_patterns("checkpoint.pt", "*.tmp", ".train.lock", "eval"))
            write_json(staging / "run.json", child)
            # All learned tensors, Adam moments/counters and both RNG states are carried over.
            state["fingerprint"] = fingerprint(child)
            torch.save(state, staging / "checkpoint.pt")
            assert checked(staging, arm) == (boundary, True)
            staging.rename(dest)
    assert read_json(dest / "run.json") == child
    paths[arm] = dest

def train_command(arm, run, stop):
    # The CLI uses the prefix config. Call its underlying trainer with the saved
    # config so the 25k hold uses exactly the settings used in the dense run.
    worker = """
import json, sys
from pathlib import Path
from recovery.training import train
run, manifest, stop = Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3])
meta = json.loads((run / 'run.json').read_text(encoding='utf-8'))
train(run, manifest, meta['protocol'], meta['seed'], 'cuda:0', meta['training_config'],
      resume=True, stop_at=stop, save_every=500)
"""
    return [sys.executable, "-u", "-c", worker, str(run), str(manifest), str(stop)]

def workers(jobs, label):
    processes = {}
    with ExitStack() as stack:
        try:
            assert len(jobs)<=len(devices)
            for lane,(arm, command) in enumerate(jobs.items()):
                gpu = devices[lane]
                worker_env = dict(env, CUDA_VISIBLE_DEVICES=gpu, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
                log = paths[arm] / f"{label}.log"
                writer = stack.enter_context(log.open("ab"))
                reader = stack.enter_context(log.open(encoding="utf-8", errors="replace"))
                reader.seek(0, os.SEEK_END)
                proc = subprocess.Popen(command, cwd=repo, env=worker_env, stdout=writer, stderr=subprocess.STDOUT)
                processes[arm] = proc, reader
                print(f"{arm} on GPU {gpu}: {label}", flush=True)
            while processes:
                finished = []
                for arm, (proc, reader) in processes.items():
                    status = proc.poll()
                    for line in reader:
                        print(f"[{arm}] {line.rstrip()}", flush=True)
                    if status not in (None, 0):
                        raise RuntimeError(f"{arm} {label} exited with status {status}. See its log.")
                    if status == 0:
                        finished.append(arm)
                for arm in finished:
                    del processes[arm]
                if processes:
                    time.sleep(2)
        finally:
            for proc, _ in processes.values():
                if proc.poll() is None:
                    proc.terminate()
            for proc, _ in processes.values():
                proc.wait()

def download(path):
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{path.name}" href="data:application/zip;base64,{encoded}">Download {path.name}</a>'))

def snapshot(tag):
    files = {"manifests/personal.json": manifest}
    if tag=="review": files["comparison.json"]=work/"comparison.json"
    records = {}
    for arm, run in paths.items():
        try:
            step, hold = checked(run, arm)
            meta = read_json(run / "run.json")
            records[arm] = dict(step=step, run_id=meta["run_id"], checkpoint_sha256=sha(run / "checkpoint.pt"),
                                lineage=meta.get("lineage"), phase="hold" if hold else "prefix")
        except (AssertionError, ValueError, KeyError, RuntimeError, OSError, EOFError) as exc:
            if tag != "diagnostics":
                raise
            records[arm] = {"checkpoint_error": f"{type(exc).__name__}: {exc}"}
        members = [run / name for name in ("checkpoint.pt", "run.json", "health.json", "initial-health.json", "training.jsonl")]
        members += list(run.glob("train-*.png")) + list(run.glob("*.log"))
        if tag in ("review", "eval-interrupted", "diagnostics"):
            members += list((run / "eval").glob("*"))
        for p in members:
            if p.is_file():
                files[(relative(arm) / p.relative_to(run)).as_posix()] = p
    receipt = dict(tag=tag, stage16_receipt=stage16_receipt, start_checkpoints=start_shas, schedule_hash=digest(plan),
                   data_hash=data_hash, arms=records,
                   devices=[torch.cuda.get_device_name(i) for i in range(2)], torch_version=str(torch.__version__),
                   cuda=torch.version.cuda, scored=tag == "review", sha256={n: sha(p) for n,p in files.items()})
    dest = outputs / f"personal-s42-stage17-{tag}.zip"
    tmp = dest.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for name, p in files.items():
            z.write(p, name)
        z.writestr("stage17-review.json", json.dumps(receipt, indent=2, allow_nan=False))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None and set(z.namelist()) == set(receipt["sha256"]) | {"stage17-review.json"}
    os.replace(tmp, dest)
    print(f"Saved {dest.name} ({dest.stat().st_size / 1e6:.1f} MB), SHA-256 {sha(dest)}", flush=True)
    return dest

In [ ]:
def finish_runs():
    dense,sparse,reg=arms
    waves=[]
    for base in (10000,30000,50000):
        mid,end=base+10000,base+20000
        waves.extend([((dense,mid),(sparse,mid)),((reg,mid),(dense,end)),((sparse,end),(reg,end))])
    waves.extend([((dense,endpoint),(sparse,endpoint)),((reg,endpoint),)])
    for i,wave in enumerate(waves,1):
        jobs={}
        for arm,stop in wave:
            step,hold=checked(paths[arm],arm)
            if step>=stop:continue
            if not healthy(paths[arm],step):
                raise RuntimeError(f"{arm} failed its restored training check at {step}.")
            if stop>boundary and not hold:fork_hold(arm)
            jobs[arm]=train_command(arm,paths[arm],stop)
        if jobs:
            workers(jobs,"training")
            for arm,stop in wave:
                if arm in jobs and (checked(paths[arm],arm)[0]!=stop or not healthy(paths[arm],stop)):
                    raise RuntimeError(f"Review {arm} at {stop}.")
            tag={3:"step30000",6:"step50000",11:"step75000"}.get(i,f"wave{i:02}")
            path=snapshot(tag)
            if i in (3,6,11):download(path)
    assert all(checked(paths[a],a)==(endpoint,True) and healthy(paths[a],endpoint) for a in arms)

try:
    finish_runs()
except BaseException:
    download(snapshot("diagnostics"))
    raise

In [ ]:
def verify_scores(arm):
    run = paths[arm]
    meta = read_json(run / "run.json")
    assert checked(run, arm) == (endpoint, True)
    result = read_json(run / "eval/metrics.json")
    compare, rows = result["comparison"], result["frames"]
    assert result["comparison_hash"] == digest(compare) == digest(expected_comparison)
    assert compare["lpips_weights_sha256"] == lpips_info["lpips_model_sha256"]
    assert [r["frame"] for r in rows] == compare["test_ids"] == [test.frame_id(i) for i in range(len(test))]
    assert (result["protocol"], result["run_id"], result["seed"], result["scene"], result["train_views"]) == (
        arm, meta["run_id"], 42, "personal", view_counts[arm])
    model = make_model(plan, torch.device("cpu"))
    model.load_state_dict(load_state(run, meta)["model"])
    assert result["model_sha256"] == model_fingerprint(model)
    keys = ("psnr", "ssim", "lpips", "white_psnr")
    for key in keys:
        values = [r[key] for r in rows]
        assert all(map(math.isfinite, values)) and math.isclose(statistics.mean(values), result["mean"][key], abs_tol=1e-8)
    assert all(math.isclose(row["white_psnr"], expected, abs_tol=1e-5) for row, expected in zip(rows,white_views))
    if arm != "dense-vanilla":
        fit = result["training_views"]
        assert [r["frame"] for r in fit["frames"]] == [frozen["frames"][i]["file_path"] for i in camera_ids]
        assert math.isfinite(fit["mean_psnr"]) and math.isclose(statistics.mean(r["psnr"] for r in fit["frames"]), fit["mean_psnr"], abs_tol=1e-8)
        assert fit["at_least_30_db"] == (fit["mean_psnr"] >= 30)
    else:
        assert "training_views" not in result
    for index in (0, 6, 11):
        for kind in ("target", "rgb", "depth", "opacity"):
            name = f"view-{index:03d}-{kind}.png"
            assert rows[index]["artifacts"][kind] == name
            with Image.open(run / "eval" / name) as pic:
                assert pic.format == "PNG" and pic.size == (240, 320)
                pic.verify()
    print(arm, result["mean"], flush=True)

def score_pair():
    jobs = {}
    for arm, run in paths.items():
        assert checked(run, arm) == (endpoint, True)
        if (run / "eval/metrics.json").is_file():
            verify_scores(arm)
        else:
            jobs[arm] = [sys.executable, "-u", "-m", "recovery.media", "eval", "--run", str(run),
                         "--manifest", str(manifest), "--device", "cuda:0", "--chunk", "1024"]
    for i in range(0,len(jobs),2):
        workers(dict(list(jobs.items())[i:i+2]), "evaluation")
    for arm in jobs:
        verify_scores(arm)
    from recovery.report import compare
    write_json(work/"comparison.json",compare([read_json(paths[a]/"eval/metrics.json") for a in arms]))
    download(snapshot("review"))

try:
    score_pair()
except BaseException:
    download(snapshot("eval-interrupted"))
    raise